# Tide PDF
## Dask

In [ ]:
# Set up a local cluster for distributed computing.
from distributed import LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
client

In [ ]:
import xarray as xr
import pandas as pd
from ipywidgets import interact, IntSlider
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from numpy.linalg import lstsq
from tqdm import tqdm
from scipy.signal import savgol_filter
import xdggs
import sparse

In [ ]:
from tide_function import *

## 1) Chargement tag avec filtre sinus 

Ouverture du tag via pangeo-fish, on réduit le time step à 1 minute à la place de 2 secondes (données originales).
Utilisation du tagging event release point pour reshaped tag. 

In [ ]:
from pangeo_fish.helpers import load_tag

bbox = {"latitude": [46, 60], "longitude": [-10, 5]}

# emplacement à CHANGER selon local ou cloud

tag, tag_log, time_slice = load_tag(
    tag_root="/home/jovyan/pangeo-fish-old/notebooks/nouveau_tag",
    tag_name="A15789",
    storage_options=None,
)

ds = tag_log["pressure"]
df_resample = ds.resample(
    {"time": "1T"}
).first()  # On prends toutes les minutes la pression
df_resample = df_resample.reset_index("time")
df = df_resample.to_dataframe()
df = df.reset_index()  # On enlève l'index time
df["time"] = (
    tag_log["time"].resample({"time": "1T"}).first()
)  # <--- Ajout de la colonne timedans df + on prends toute les minutes

In [ ]:
tag_test = tide_behav_extr(
    df=df,
    tagno="A15789",
    tideFL=1,
    tideLV=[0.42, 0.85, 0.6],
    behavFL=16,
    behavLV=[0.42, 0.85, 0.6],
    dt=1,
)

tag_test

### Plot des instants de marée

In [ ]:
# Dataset : ds
time = tag_test["time"].values
depth = tag_test["depth"].values
tide_found = tag_test["tide_found"].values  # 0 ou 1

# Création du plot
plt.figure(figsize=(15, 4))
plt.plot(time, depth, color="blue", label="Profondeur")

# Ajouter un fond vert pour les périodes de marée
# On va utiliser fill_between pour les zones où tide_found==1
plt.fill_between(
    time,
    depth.min(),
    depth.max(),
    where=(tide_found == 1),
    color="green",
    alpha=0.3,
    label="Marée détectée",
)

plt.xlabel("Temps")
plt.ylabel("Profondeur (m)")
plt.title(f'Temporal Depth & Tidal Detection - Tag {tag_test.attrs["tagno"]}')
plt.legend()
plt.tight_layout()
plt.show()

**Plot intéractif pour zoomer sur certaine tranche**

In [ ]:
from ipywidgets import interact, IntSlider


def plot_tide_segment(start_idx):
    end_idx = start_idx + 500  # Ajuste la taille de la fenêtre
    plt.figure(figsize=(15, 4))
    plt.plot(
        time[start_idx:end_idx],
        depth[start_idx:end_idx],
        color="blue",
        label="Profondeur",
    )
    plt.fill_between(
        time[start_idx:end_idx],
        depth.min(),
        depth.max(),
        where=(tide_found[start_idx:end_idx] == 1),
        color="green",
        alpha=0.3,
        label="Marée détectée",
    )
    plt.xlabel("Temps")
    plt.ylabel("Profondeur (m)")
    plt.title(f"Segment {start_idx} à {end_idx}")
    plt.legend()
    plt.show()


# Crée un slider pour choisir le début de la plage
interact(
    plot_tide_segment,
    start_idx=IntSlider(min=0, max=len(time) - 200, step=300, value=0),
)

### On enregistre le fichier sinus sous zarr

Possibilité de changer l'emplacement pour le mettre sur S3

In [ ]:
tag_test.to_zarr(
    "/home/jovyan/pangeo-fish/notebooks/Tide/A15789_tide_behav.zarr", mode="w"
)

## 2) Chargement de la base de donnée, calcul de la vraissemblance 

Ouverture dataset comportement, tide detected  

In [ ]:
"gfts-ifremer/eel/tag/raw/Subset_NS_EC_forTMD30_21_2_2025.nc"

tide_behav = xr.open_dataset("A15789_tide_behav.zarr", engine="zarr")
tide_behav

**Base de donnée de marée fournie par MOKO à besoin d'être reshaped en [-180,180]**

In [ ]:
data_360 = xr.open_dataset("Subset_NS_EC_forTMD30_21_2_2025.nc")

data = convert_lon_360_to_180(data_360, bbox)
data

## PDF uniquement sur les instants de marée (fonction voir tide_function.py)

La cellule qui suit peut être améliorée via les fonctions pangeo. Simple reshaped horaire des indicateurs calculés précédemment.

In [ ]:
ds = tide_behav.copy()

# Convertir la coordonnée 'time' en datetime pandas
time = pd.to_datetime(ds["time"].values)

# Créer un DataFrame temporaire pour manipuler facilement les regroupements
tmp_df = pd.DataFrame(
    {
        "depth": ds["depth"].values,
        "tide_found": ds["tide_found"].values,
    },
    index=time,
)

# Créer une colonne 'hour' arrondie à l'heure
tmp_df["hour"] = tmp_df.index.floor("H")

# Propager tide_found à toute l'heure si détecté au moins une fois
tmp_df["hour_include"] = tmp_df["hour"]
for h, group in tmp_df.groupby("hour"):
    if group["tide_found"].any():
        tmp_df.loc[group.index, "hour_include"] = h

# Regrouper les mesures par "hour_include"
grouped_obs = {
    hour: group[["depth"]].values for hour, group in tmp_df.groupby("hour_include")
}

# Trier les heures
hours_sorted = sorted(grouped_obs.keys())

# td_time : heures depuis le début
td_time = np.array([(h - hours_sorted[0]).total_seconds() / 3600 for h in hours_sorted])

# td_depth : matrice (n_times, n_meas_max)
n_times = len(hours_sorted)
n_meas_max = max(len(grouped_obs[h]) for h in hours_sorted)
td_depth = np.full((n_times, n_meas_max), np.nan)
for i, h in enumerate(hours_sorted):
    vals = grouped_obs[h].flatten()
    td_depth[i, : len(vals)] = vals

# Création du vecteur tide_found pour chaque "heure incluse"
tide_found_hourly = np.array(
    [tmp_df[tmp_df["hour_include"] == h]["tide_found"].any() for h in hours_sorted],
    dtype=int,
)

### Calcul de la likelihood

In [ ]:
Likelihood = datalikelihood_tide_only_full_normalized(
    td_depth, td_time, data, tide_found_hourly, sigma_tid=2.0
)

In [ ]:
Likelihood

In [ ]:
# Save in zarr
Likelihood.to_zarr("A15789_tide_pdf.zarr", mode="w")

## 3) Regrided in Healpix 

In [ ]:
import xarray as xr

LIK = xr.open_dataset("A15789_tide_pdf.zarr", engine="zarr").compute()
LIK

In [ ]:
from pangeo_fish.helpers import open_diff_dataset, regrid_dataset

refinement_level = 12
# LIK.rename_dims('lat':latitude,'lon':longitude)
reshaped = regrid_dataset(
    ds=LIK, refinement_level=refinement_level, min_vertices=1, dims=["cells"]
)[0]
reshaped

In [ ]:
# Permet d'éviter des problèmes au moment du save / compute
var = reshaped["ocean_mask"].compute()
if isinstance(var.data, sparse.SparseArray):
    reshaped["ocean_mask"] = var.copy(data=var.data.todense())
reshaped

**Problème pour compute (trop volumineux) donc calcul par étape, il existe peut être des méthodes plus propres.** 

In [ ]:
ds = reshaped  # ton dataset
pdf = ds.pdf

chunk_size = 200
n_time = pdf.sizes["time"]

pdf_blocks = []

for start in range(0, n_time, chunk_size):
    end = min(start + chunk_size, n_time)
    print(f"Computing time slice {start}:{end}")

    block = pdf.isel(time=slice(start, end)).compute()
    pdf_blocks.append(block)

# Reconstruire la variable pdf complète
pdf_full = xr.concat(pdf_blocks, dim="time")

# Refaire un dataset identique
ds_full = ds.drop_vars("pdf").assign(pdf=pdf_full)

print("DONE — Dataset computed en entier !")

In [ ]:
ds_full

### Save du dataset 

In [ ]:
ds_full.to_zarr("A15789_tide_pdf_healpix.zarr", mode="w")

In [ ]:
ds_full = xr.open_dataset("A15789_tide_pdf_healpix.zarr", engine="zarr").compute()

### Plot

In [ ]:
ds_full.pdf.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8)

In [ ]:
(ds_full["pdf"].where(ds_full["pdf"] > 0)).min().item()

In [ ]:
ds_norme["pdf"] = ds_norme.pdf / ds_norme.pdf.sum(dim="cells")

In [ ]:
ds_norme["pdf"].sum()

# Autre recherche mais pas important 

In [ ]:
import matplotlib.pyplot as plt

t_idx = 0  # par exemple le premier pas de temps
lik_2d = LIK.likelihood.isel(time=t_idx)

plt.figure(figsize=(8, 6))
plt.pcolormesh(LIK.longitude, LIK.latitude, lik_2d, shading="auto", cmap="viridis")
plt.colorbar(label="Likelihood")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"Likelihood à time index {t_idx}")
plt.show()

In [ ]:
tag_test = xr.open_dataset("A15789_tide_pdf_healpix_h.zarr", engine="zarr").compute()
tag_test

In [ ]:
tag_test["likelihood"].isel(time=0).compute()

In [ ]:
ds_full

In [ ]:
ds_full.pdf.isel(time=slice(0, 1000)).compute().dggs.decode(
    {"grid_name": "healpix", "level": 11, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8)

In [ ]:
# fonction avec uniquement deux plots pdf + données brutes


def plot_lik_and_td_depth(t_idx):
    fig = plt.figure(figsize=(12, 10))

    # --------- Axe pour LIK avec Cartopy ---------
    ax1 = plt.subplot(2, 1, 1, projection=ccrs.PlateCarree())
    ax1.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax1.add_feature(cfeature.LAND, facecolor="lightgray")
    ax1.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax1.add_feature(cfeature.OCEAN, facecolor="lightblue")

    mesh = ax1.pcolormesh(
        lon_grid,
        lat_grid,
        LIK_bbox_full[t_idx],
        shading="auto",
        cmap="viridis",
        transform=ccrs.PlateCarree(),
    )
    plt.colorbar(mesh, ax=ax1, orientation="vertical", label="Likelihood")
    ax1.set_title(f"Likelihood 2D au temps t={td_time[t_idx]:.2f} h")

    # --------- Axe pour td_depth ---------
    ax2 = plt.subplot(2, 1, 2)
    td_current = td_depth[t_idx, :]  # shape (n_meas,)
    td_smoothed = savgol_filter(depth, window_length=15, polyorder=2)
    meas_idx = np.arange(len(td_current))
    ax2.plot(meas_idx, td_current, "bo", alpha=0.7)
    ax2.set_xlabel("Mesure index")
    ax2.set_ylabel("Tag depth [m]")
    ax2.set_title(f"Tag depth pour l'heure t_idx={t_idx} ({td_time[t_idx]:.2f} h)")
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
lon_min, lon_max = -10, 10
lat_min, lat_max = 48, 59
lon_vals = data.lon.values
lat_vals = data.lat.values
# Masques
lon_mask = (lon_vals >= lon_min) & (lon_vals <= lon_max)
lat_mask = (lat_vals >= lat_min) & (lat_vals <= lat_max)

# échelle temporelle
time_hourly = time[::60]  # vrai timestamp pour chaque t_idx

# Sélection directe avec les masques booléens
LIK_bbox_full = LIK[:, lat_mask, :][
    :, :, lon_mask
]  # shape = (n_times, n_lat_bbox, n_lon_bbox)

# Grille correspondante
lon_grid, lat_grid = np.meshgrid(lon_vals[lon_mask], lat_vals[lat_mask])
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np


def plot_lik_and_td_depth(t_idx):
    fig = plt.figure(figsize=(12, 14))

    ########## ----- AXE 1 : LIKELIHOOD 2D ----- ##########
    ax1 = plt.subplot(2, 1, 1, projection=ccrs.PlateCarree())
    ax1.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax1.add_feature(cfeature.LAND, facecolor="lightgray")
    ax1.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax1.add_feature(cfeature.OCEAN, facecolor="lightblue")

    mesh = ax1.pcolormesh(
        lon_grid,
        lat_grid,
        LIK_bbox_full[t_idx],
        shading="auto",
        cmap="viridis",
        transform=ccrs.PlateCarree(),
    )
    plt.colorbar(mesh, ax=ax1, orientation="vertical", label="Likelihood")

    ax1.set_title(f"Likelihood 2D au temps réel = {time_hourly[t_idx]}")

    ########## ----- AXE 2 : depth brut ----- ##########
    ax2 = plt.subplot(2, 1, 2)
    td_current = td_depth[t_idx, :]  # shape: (n_meas,)
    depth_smoothed = savgol_filter(td_current, window_length=15, polyorder=2)
    meas_idx = np.arange(len(td_current))

    ax2.plot(meas_idx, td_current, "bo", alpha=0.7, label="Raw depth")
    ax2.plot(depth_smoothed, "-", linewidth=1.5, label="Smoothed depth")
    ax2.set_xlabel("Mesure index")
    ax2.set_ylabel("Tag depth [m]")
    ax2.set_title(f"Tag depth (brut) – t_idx={t_idx} ({time_hourly[t_idx]})")
    ax2.grid(True)
    ax2.legend()

    plt.tight_layout()
    plt.show()


interact(
    plot_lik_and_td_depth, t_idx=IntSlider(min=0, max=len(td_time) - 1, step=1, value=0)
)

### Here we load tag

In [ ]:
import pandas as pd

tag_test = pd.read_csv(
    ("~/pangeo-fish/notebooks/nouveau_tag/A15789/dst.csv"),
    parse_dates=["time"],
    index_col="time",
).tz_convert(None)
tag_test

import hvplot.pandas

In [ ]:
import pandas as pd

# S'assurer que l'index est datetime
tag_test.index = pd.to_datetime(tag_test.index)

# Créer une nouvelle colonne 'hour' pour regrouper par heure
tag_test["hour"] = tag_test.index.floor("H")  # arrondi à l'heure

# Créer un dictionnaire avec key = heure et value = les observations dans cette heure
grouped_obs = {
    hour: group[["pressure"]].values for hour, group in tag_test.groupby("hour")
}

# Exemple : observations de la première heure
first_hour = list(grouped_obs.keys())[0]
# Trier les heures
hours_sorted = sorted(grouped_obs.keys())

# td_time : heures depuis le début en float (ex : heures)
td_time = np.array([(h - hours_sorted[0]).total_seconds() / 3600 for h in hours_sorted])

# td_depth : array (n_times, n_meas_max)
# Comme chaque heure peut avoir un nombre différent de mesures, on va "padding" avec NaN
n_times = len(hours_sorted)
n_meas_max = max(len(grouped_obs[h]) for h in hours_sorted)
td_depth2 = np.full((n_times, n_meas_max), np.nan)

for i, h in enumerate(hours_sorted):
    vals = grouped_obs[h].flatten()  # toutes les mesures de cette heure
    td_depth2[i, : len(vals)] = vals

In [ ]:
import pandas as pd
import numpy as np


def bin_measurements(df, freq="H", col="pressure"):
    """
    Regroupe les mesures par heure ou par jour et renvoie td_time et td_depth.

    Args:
        df (pd.DataFrame): DataFrame avec un index datetime et une colonne à grouper.
        freq (str): 'H' pour heure, 'D' pour jour.
        col (str): nom de la colonne à grouper.

    Returns:
        td_time (np.ndarray): temps depuis le début (en heures).
        td_depth (np.ndarray): mesures avec padding NaN.
        bins (pd.DatetimeIndex): heures ou jours correspondant aux lignes de td_depth.
    """
    # S'assurer que l'index est datetime
    df.index = pd.to_datetime(df.index)

    # Grouper par fréquence
    grouped = df.groupby(pd.Grouper(freq=freq))[col].apply(list)
    bins = grouped.index.sort_values()

    # td_time : temps depuis le début en heures
    td_time = (bins - bins[0]).total_seconds() / 3600

    # td_depth : padding avec NaN
    n_times = len(bins)
    n_meas_max = grouped.apply(len).max()
    td_depth = np.full((n_times, n_meas_max), np.nan)

    for i, vals in enumerate(grouped.loc[bins]):
        td_depth[i, : len(vals)] = vals

    return td_time, td_depth, bins


# Exemple d'utilisation :
td_time_hour, td_depth_hour, bins_hour = bin_measurements(tag_test, freq="H")
# td_time_day, td_depth_day, bins_day = bin_measurements(tag_test, freq='D')

In [ ]:
LIK_hour = datalikelihood_fast_xr_multi_vec_mask(
    td_depth_hour, td_time_hour.to_numpy(dtype=float), tidal_ds=data, sigma_tid=5.0
)

In [ ]:
import pickles

In [ ]:
LIK_day = datalikelihood_fast_xr_multi_vec_mask(
    td_depth_day, td_time_day.to_numpy(dtype=float), tidal_ds=data, sigma_tid=5.0
)

In [ ]:
# -------------------
# Sélection zone Europe/Nord Atlantique
# -------------------
lon_min, lon_max = 350, 360
lat_min, lat_max = 48, 59
lon_vals = data.lon.values
lat_vals = data.lat.values
# Masques
lon_mask = (lon_vals >= lon_min) & (lon_vals <= lon_max)
lat_mask = (lat_vals >= lat_min) & (lat_vals <= lat_max)

# Sélection directe avec les masques booléens
LIK_bbox = LIK_hour[:, lat_mask, :][
    :, :, lon_mask
]  # shape = (n_times, n_lat_bbox, n_lon_bbox)

# Grille correspondante
lon_grid, lat_grid = np.meshgrid(lon_vals[lon_mask], lat_vals[lat_mask])


# -------------------
# Fonction de tracé interactif
# -------------------
def plot_lik_timestep(t_idx):
    fig = plt.figure(figsize=(10, 8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN, facecolor="lightblue")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    mesh = ax.pcolormesh(
        lon_grid,
        lat_grid,
        LIK_bbox[t_idx],
        shading="auto",
        cmap="viridis",
        transform=ccrs.PlateCarree(),
    )
    plt.colorbar(mesh, ax=ax, orientation="vertical", label="Likelihood")
    plt.title(f"Likelihood 2D au temps t={td_time[t_idx]}")
    plt.show()


# Slider interactif
interact(
    plot_lik_timestep, t_idx=IntSlider(min=0, max=len(td_time) - 1, step=1, value=0)
)

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import numpy as np


def plot_lik_and_td_depth(t_idx):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))

    # --------- Plot LIK ---------
    mesh = ax1.pcolormesh(
        lon_grid, lat_grid, LIK_bbox[t_idx], shading="auto", cmap="viridis"
    )
    ax1.set_title(f"Likelihood 2D au temps t={td_time[t_idx]:.2f} h")
    ax1.set_xlabel("Longitude")
    ax1.set_ylabel("Latitude")
    plt.colorbar(mesh, ax=ax1, orientation="vertical", label="Likelihood")

    # --------- Plot td_depth pour l'heure t_idx ---------
    td_current = td_depth[t_idx, :]  # shape (n_meas,)
    meas_idx = np.arange(len(td_current))
    ax2.plot(meas_idx, td_current, "bo", alpha=0.7)
    ax2.set_xlabel("Mesure index")
    ax2.set_ylabel("Tag depth [m]")
    ax2.set_title(f"Tag depth pour l'heure t_idx={t_idx} ({td_time[t_idx]:.2f} h)")
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


# Slider interactif
interact(
    plot_lik_and_td_depth, t_idx=IntSlider(min=0, max=len(td_time) - 1, step=1, value=0)
)

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import numpy as np

# td_depth : shape (n_times, n_meas)
# td_time : temps en heures
# LIK_bbox : shape (n_times, n_lat, n_lon)
# lon_grid, lat_grid : grille de la zone


def plot_lik_and_depth(t_idx):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # --------- Plot LIK ---------
    mesh = ax1.pcolormesh(
        lon_grid, lat_grid, LIK_bbox[t_idx], shading="auto", cmap="viridis"
    )
    ax1.set_title(f"Likelihood 2D au temps t={td_time[t_idx]:.2f} h")
    ax1.set_xlabel("Longitude")
    ax1.set_ylabel("Latitude")
    plt.colorbar(mesh, ax=ax1, orientation="vertical", label="Likelihood")

    # --------- Plot td_depth ---------
    n_meas = td_depth.shape[1]
    for m in range(n_meas):
        ax2.plot(td_time, td_depth[:, m], color="tab:blue", alpha=0.5)

    # Ligne verticale indiquant le temps courant
    ax2.axvline(
        x=td_time[t_idx],
        color="red",
        linestyle="--",
        linewidth=2,
        label="Current timestep",
    )
    ax2.set_xlabel("Time [hours]")
    ax2.set_ylabel("Tag depth [m]")
    ax2.set_title("Tag depth vs time")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


# Slider interactif
interact(
    plot_lik_and_depth, t_idx=IntSlider(min=0, max=len(td_time) - 1, step=1, value=0)
)

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import numpy as np

# td_depth : shape (n_times, n_meas)
# td_time : temps en heures
# LIK_bbox : shape (n_times, n_lat, n_lon)
# lon_grid, lat_grid : grille de la zone


def plot_lik_and_depth(t_idx):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # --------- Plot LIK ---------
    mesh = ax1.pcolormesh(
        lon_grid, lat_grid, LIK_bbox[t_idx], shading="auto", cmap="viridis"
    )
    ax1.set_title(f"Likelihood 2D au temps t={td_time[t_idx]:.2f} h")
    ax1.set_xlabel("Longitude")
    ax1.set_ylabel("Latitude")
    plt.colorbar(mesh, ax=ax1, orientation="vertical", label="Likelihood")

    # --------- Plot td_depth ---------
    n_meas = td_depth.shape[1]
    for m in range(n_meas):
        ax2.plot(td_time, td_depth[:, m], color="tab:blue", alpha=0.5)

    # Ligne verticale indiquant le temps courant
    ax2.axvline(
        x=td_time[t_idx],
        color="red",
        linestyle="--",
        linewidth=2,
        label="Current timestep",
    )
    ax2.set_xlabel("Time [hours]")
    ax2.set_ylabel("Tag depth [m]")
    ax2.set_title("Tag depth vs time")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


# Slider interactif
interact(
    plot_lik_and_depth, t_idx=IntSlider(min=0, max=len(td_time) - 1, step=1, value=0)
)

In [ ]:
import hvplot.xarray
import hvplot.pandas
import holoviews as hv

hv.extension("bokeh")

# LIK_da et td_depth_melt préparés comme avant

# Carte interactive
lik_plot = LIK_da.hvplot.image(
    x="lon",
    y="lat",
    kdims=["lon", "lat"],
    vdims="time",
    groupby="time",
    cmap="viridis",
    width=500,
    height=500,
)

# Série temporelle de pression
pressure_plot = td_depth_melt.hvplot.line(
    x="time", y="pressure", hue="measurement", width=400, height=500
)

# Combine les plots côte à côte
combined = lik_plot + pressure_plot

# Dernière ligne de la cellule => affichage automatique
combined

### COV calculated covariance full with depth (still mean of depth)

In [ ]:
LIK_full_hour = datalikelihood_full_xr_vec(
    td_depth_hour, td_time_hour.to_numpy(dtype=float), tidal_ds=data, sigma_tid=5.0
)

In [ ]:
LIK_full_day = datalikelihood_full_xr_vec(
    td_depth_day, td_time_day.to_numpy(dtype=float), tidal_ds=data, sigma_tid=5.0
)

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import numpy as np

# -------------------
# Sélection zone Europe/Nord Atlantique
# -------------------
lon_min, lon_max = -10, 10
lat_min, lat_max = 48, 59
lon_vals = data.lon.values
lat_vals = data.lat.values
# Masques
lon_mask = (lon_vals >= lon_min) & (lon_vals <= lon_max)
lat_mask = (lat_vals >= lat_min) & (lat_vals <= lat_max)

# Sélection directe avec les masques booléens
LIK_bbox = LIK_full_hour[:, lat_mask, :][:, :, lon_mask]
lon_grid, lat_grid = np.meshgrid(lon_vals[lon_mask], lat_vals[lat_mask])


def plot_lik_and_td_depth(t_idx):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))

    # --------- Plot LIK ---------
    mesh = ax1.pcolormesh(
        lon_grid, lat_grid, LIK_bbox[t_idx], shading="auto", cmap="viridis"
    )
    ax1.set_title(f"Likelihood 2D au temps t={td_time[t_idx]:.2f} h")
    ax1.set_xlabel("Longitude")
    ax1.set_ylabel("Latitude")
    plt.colorbar(mesh, ax=ax1, orientation="vertical", label="Likelihood")

    # --------- Plot td_depth pour l'heure t_idx ---------
    td_current = td_depth[t_idx, :]  # shape (n_meas,)
    meas_idx = np.arange(len(td_current))
    ax2.plot(meas_idx, td_current, "bo", alpha=0.7)
    ax2.set_xlabel("Mesure index")
    ax2.set_ylabel("Tag depth [m]")
    ax2.set_title(f"Tag depth pour l'heure t_idx={t_idx} ({td_time[t_idx]:.2f} h)")
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


# Slider interactif
interact(
    plot_lik_and_td_depth, t_idx=IntSlider(min=0, max=len(td_time) - 1, step=1, value=0)
)

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import numpy as np

# -------------------
# Sélection zone Europe/Nord Atlantique
# -------------------
lon_min, lon_max = 350, 360
lat_min, lat_max = 48, 59
lon_vals = data.lon.values
lat_vals = data.lat.values
# Masques
lon_mask = (lon_vals >= lon_min) & (lon_vals <= lon_max)
lat_mask = (lat_vals >= lat_min) & (lat_vals <= lat_max)

# Sélection directe avec les masques booléens
LIK_bbox_full = LIK_full_hour[:, lat_mask, :][
    :, :, lon_mask
]  # shape = (n_times, n_lat_bbox, n_lon_bbox)

# Grille correspondante
lon_grid, lat_grid = np.meshgrid(lon_vals[lon_mask], lat_vals[lat_mask])
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np


def plot_lik_and_td_depth(t_idx):
    fig = plt.figure(figsize=(12, 10))

    # --------- Axe pour LIK avec Cartopy ---------
    ax1 = plt.subplot(2, 1, 1, projection=ccrs.PlateCarree())
    ax1.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax1.add_feature(cfeature.LAND, facecolor="lightgray")
    ax1.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax1.add_feature(cfeature.OCEAN, facecolor="lightblue")

    mesh = ax1.pcolormesh(
        lon_grid,
        lat_grid,
        LIK_bbox_full[t_idx],
        shading="auto",
        cmap="viridis",
        transform=ccrs.PlateCarree(),
    )
    plt.colorbar(mesh, ax=ax1, orientation="vertical", label="Likelihood")
    ax1.set_title(f"Likelihood 2D au temps t={td_time[t_idx]:.2f} h")

    # --------- Axe pour td_depth ---------
    ax2 = plt.subplot(2, 1, 2)
    td_current = td_depth[t_idx, :]  # shape (n_meas,)
    meas_idx = np.arange(len(td_current))
    ax2.plot(meas_idx, td_current, "bo", alpha=0.7)
    ax2.set_xlabel("Mesure index")
    ax2.set_ylabel("Tag depth [m]")
    ax2.set_title(f"Tag depth pour l'heure t_idx={t_idx} ({td_time[t_idx]:.2f} h)")
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


interact(
    plot_lik_and_td_depth, t_idx=IntSlider(min=0, max=len(td_time) - 1, step=1, value=0)
)

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np


def plot_lik_and_td_depth(t_idx):
    # Figure plus grande
    fig = plt.figure(figsize=(10, 15))

    # Grille pour deux axes : plus de place pour LIK
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 0.5], hspace=0.3)

    # --------- Axe pour LIK avec Cartopy ---------
    ax1 = fig.add_subplot(gs[0], projection=ccrs.PlateCarree())
    ax1.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax1.add_feature(cfeature.LAND, facecolor="lightgray")
    ax1.add_feature(cfeature.COASTLINE, linewidth=1.0)
    ax1.add_feature(cfeature.OCEAN, facecolor="lightblue")

    mesh = ax1.pcolormesh(
        lon_grid,
        lat_grid,
        LIK_bbox_full[t_idx],
        shading="auto",
        cmap="viridis",
        transform=ccrs.PlateCarree(),
    )
    cbar = plt.colorbar(mesh, ax=ax1, orientation="vertical", fraction=0.04, pad=0.02)
    cbar.set_label("Likelihood", fontsize=12)
    ax1.set_title(f"Likelihood 2D au temps t={td_time[t_idx]:.2f} h", fontsize=14)

    # --------- Axe pour td_depth ---------
    ax2 = fig.add_subplot(gs[1])
    td_current = td_depth[t_idx, :]  # shape (n_meas,)
    meas_idx = np.arange(len(td_current))
    ax2.plot(meas_idx, td_current, "bo", alpha=0.7)
    ax2.set_xlabel("Mesure index", fontsize=12)
    ax2.set_ylabel("Tag depth [m]", fontsize=12)
    ax2.set_title(
        f"Tag depth pour l'heure t_idx={t_idx} ({td_time[t_idx]:.2f} h)", fontsize=14
    )
    ax2.grid(True)

    plt.show()


interact(
    plot_lik_and_td_depth, t_idx=IntSlider(min=0, max=len(td_time) - 1, step=1, value=0)
)

### Checking diff between fast and full

In [ ]:
# -------------------
# Sélection zone Europe/Nord Atlantique
# -------------------
lon_min, lon_max = 350, 360
lat_min, lat_max = 48, 59
lon_vals = data.lon.values
lat_vals = data.lat.values
# Masques
lon_mask = (lon_vals >= lon_min) & (lon_vals <= lon_max)
lat_mask = (lat_vals >= lat_min) & (lat_vals <= lat_max)

# Sélection directe avec les masques booléens
LIK_bbox = LIK_full_day[:, lat_mask, :][
    :, :, lon_mask
]  # shape = (n_times, n_lat_bbox, n_lon_bbox)

# Grille correspondante
lon_grid, lat_grid = np.meshgrid(lon_vals[lon_mask], lat_vals[lat_mask])


# -------------------
# Fonction de tracé interactif
# -------------------
def plot_lik_timestep(t_idx):
    fig = plt.figure(figsize=(10, 8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN, facecolor="lightblue")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    mesh = ax.pcolormesh(
        lon_grid,
        lat_grid,
        LIK_bbox[t_idx],
        shading="auto",
        cmap="viridis",
        transform=ccrs.PlateCarree(),
    )
    plt.colorbar(mesh, ax=ax, orientation="vertical", label="Likelihood")
    plt.title(f"Likelihood 2D au temps t={td_time[t_idx]}")
    plt.show()


# Slider interactif
interact(
    plot_lik_timestep, t_idx=IntSlider(min=0, max=len(td_time) - 1, step=1, value=0)
)